# Probability Basics

Probability is a formalised way of counting — specifically, counting how many outcomes match a condition out of all possible outcomes. You already do this in code:

```python
# Empirical probability: what fraction of users clicked?
p_click = len([u for u in users if u.clicked]) / len(users)

# Uniform probability: what fraction of inputs satisfy a condition?
p_even = len([n for n in range(1, 7) if n % 2 == 0]) / 6  # rolling even on a die
```

This notebook covers the foundations: what probability is, how to build and query a sample space, and two tools for understanding what happens over many trials — the Law of Large Numbers and standard deviation.

**Prerequisites:** Notebooks 00–06. Basic fractions.

**This is part 1 of 3.** Part 2 covers compound events and independence. Part 3 covers conditional probability and Bayes' theorem.

---

## What is Probability?

Probability is a number between 0 and 1 that measures how likely an event is to occur.

- `0` means impossible
- `1` means certain
- `0.5` means equally likely to happen or not

For a **uniform** process — one where every outcome is equally likely — the formula is straightforward:

$$P(A) = \frac{\text{number of outcomes in } A}{\text{total number of possible outcomes}}$$

This is exactly `len(event) / len(sample_space)` in Python.

In [ ]:
# P(rolling a 4 on a fair six-sided die)
sample_space = {1, 2, 3, 4, 5, 6}
event_roll_4 = {4}

p = len(event_roll_4) / len(sample_space)
print(f"P(rolling a 4) = {len(event_roll_4)}/{len(sample_space)} = {p:.4f}")

# Same pattern works for any event
event_even  = {2, 4, 6}
event_gt4   = {5, 6}
event_prime = {2, 3, 5}

for name, event in [("even", event_even), ("> 4", event_gt4), ("prime", event_prime)]:
    p = len(event) / len(sample_space)
    print(f"P({name}) = {len(event)}/{len(sample_space)} = {p:.4f}")

---

## Sample Space and Events

The **sample space** is the set of all possible outcomes. An **event** is any subset of the sample space — the outcomes you care about.

This maps directly to sets in Python. The sample space is the full set; an event is a filtered subset. Probability is `len(event) / len(sample_space)`.

A few concrete examples:

In [ ]:
def probability(event, sample_space):
    """P(event) assuming uniform probability over sample_space."""
    return len(event) / len(sample_space)

# Coin flip
coin = {'H', 'T'}
print(f"P(Heads) = {probability({'H'}, coin)}")

# Standard deck of cards
suits  = ['H', 'D', 'C', 'S']
values = ['2','3','4','5','6','7','8','9','10','J','Q','K','A']
deck   = {(v, s) for v in values for s in suits}

aces       = {card for card in deck if card[0] == 'A'}
red_cards  = {card for card in deck if card[1] in ('H', 'D')}
face_cards = {card for card in deck if card[0] in ('J', 'Q', 'K')}

print(f"P(Ace)       = {len(aces)}/{len(deck)} = {probability(aces, deck):.4f}")
print(f"P(Red)       = {len(red_cards)}/{len(deck)} = {probability(red_cards, deck):.4f}")
print(f"P(Face card) = {len(face_cards)}/{len(deck)} = {probability(face_cards, deck):.4f}")

---

## The Law of Large Numbers

Theoretical probability describes what *should* happen in the long run. In a small number of trials, results can vary wildly — that's expected, not a bug. As trials accumulate, observed frequencies converge toward the theoretical probability.

This convergence is the **Law of Large Numbers**. It is why probability theory is useful in practice: given enough data, empirical frequencies are reliable estimates of the underlying probabilities.

The key word is *enough*. Small samples are inherently noisy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Small samples: each of 6 independent samples of 10 rolls — all look different
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.flatten()

for i in range(6):
    np.random.seed(i)
    rolls  = np.random.randint(1, 7, size=10)
    counts = np.bincount(rolls, minlength=7)[1:]
    probs  = counts / 10

    ax = axes[i]
    ax.bar(range(1, 7), probs, color='lightcoral', edgecolor='black')
    ax.axhline(1/6, color='red', linestyle='--', alpha=0.7, label='True prob (1/6)')
    ax.set_xticks(range(1, 7))
    ax.set_title(f'Sample {i+1} — 10 rolls')
    ax.set_xlabel('Die face')
    ax.set_ylabel('Observed probability')
    ax.set_ylim(0, 0.5)
    ax.legend(fontsize=8)

plt.suptitle('Small samples are noisy: 6 independent sets of 10 rolls', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Same sequence of rolls, progressively larger windows — watch convergence
np.random.seed(42)
rolls = np.random.randint(1, 7, size=20_000)

milestones = [10, 50, 200, 1000, 5000, 20000]

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey=True)
axes = axes.flatten()

for i, n in enumerate(milestones):
    sample = rolls[:n]
    counts = np.bincount(sample, minlength=7)[1:]
    probs  = counts / n

    ax = axes[i]
    ax.bar(range(1, 7), probs, color='skyblue', edgecolor='black')
    ax.axhline(1/6, color='red', linestyle='--', alpha=0.7, label='True prob (1/6)')
    ax.set_xticks(range(1, 7))
    ax.set_title(f'After {n:,} rolls')
    ax.set_xlabel('Die face')
    ax.set_ylabel('Observed probability')
    ax.set_ylim(0, 0.5)
    ax.legend(fontsize=8)

plt.suptitle('Law of Large Numbers: observed frequencies converge to true probabilities', fontsize=14)
plt.tight_layout()
plt.show()

---

## Variance and Standard Deviation: Measuring Spread

The Law of Large Numbers tells you that outcomes average out over time. But averages hide something important: how much individual outcomes *scatter* around that average.

Think of two servers: one that responds in exactly 100ms every time, and one that alternates between 10ms and 190ms. Same mean. Very different behaviour. The second is much harder to depend on.

**Variance** quantifies that scatter. It is the average squared distance from the mean. Squaring ensures that deviations above and below both count positively — they don't cancel out:

$$\sigma^2 = \frac{\sum_{i=1}^{n} (x_i - \bar{x})^2}{n}$$

**Standard deviation** is the square root of variance — which brings the units back to match the original data (milliseconds, not milliseconds²):

$$\sigma = \sqrt{\sigma^2}$$

A high standard deviation means outcomes are spread wide. A low one means they cluster tightly around the mean.

In [ ]:
import math
import numpy as np

def mean(data):
    return sum(data) / len(data)

def variance(data):
    m = mean(data)
    return sum((x - m) ** 2 for x in data) / len(data)

def std_dev(data):
    return math.sqrt(variance(data))

consistent   = [98, 100, 99, 101, 100, 100, 99, 101]  # response times (ms)
inconsistent = [10, 190, 20, 180, 50, 150, 80, 120]

for label, data in [("Consistent", consistent), ("Inconsistent", inconsistent)]:
    m  = mean(data)
    v  = variance(data)
    sd = std_dev(data)
    print(f"{label}:")
    print(f"  Data:    {data}")
    print(f"  Mean:    {m:.1f} ms")
    print(f"  Var:     {v:.1f} ms²")
    print(f"  Std dev: {sd:.1f} ms")
    print()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

consistent   = [98, 100, 99, 101, 100, 100, 99, 101]
inconsistent = [10, 190, 20, 180, 50, 150, 80, 120]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, data, label in zip(axes, [consistent, inconsistent], ['Consistent', 'Inconsistent']):
    m  = np.mean(data)
    sd = np.std(data)
    xs = range(len(data))

    ax.plot(xs, data, 'o-', color='steelblue', linewidth=2, markersize=7, label='Response time')
    ax.axhline(m,      color='red',    linestyle='--', linewidth=1.5, label=f'Mean = {m:.0f}')
    ax.axhline(m + sd, color='orange', linestyle=':',  linewidth=1.5, label=f'+1 std = {m+sd:.0f}')
    ax.axhline(m - sd, color='orange', linestyle=':',  linewidth=1.5, label=f'-1 std = {m-sd:.0f}')

    for i, y in enumerate(data):
        ax.plot([i, i], [m, y], color='gray', linewidth=0.8, alpha=0.5)

    ax.set_title(f'{label} (std dev = {sd:.1f} ms)', fontsize=11)
    ax.set_xlabel('Request #')
    ax.set_ylabel('Response time (ms)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Same mean, different spread — standard deviation makes the difference visible', fontsize=12)
plt.tight_layout()
plt.show()

---

## Exercises

1. Build the sample space for rolling **two** dice (pairs of values). What is `P(sum = 7)`? What is `P(sum = 12)`?
2. In the LLN visualisation, change `size=20_000` to `size=100`. Does it still converge? What does that tell you?
3. Compute the variance and standard deviation of `[1, 1, 1, 1, 100]` by hand, then verify in code. Why is the result so large?
4. Two processes have the same mean of 50. Process A has std dev 2. Process B has std dev 20. Without running code, describe what each looks like on a chart.

In [ ]:
# Your experiments here


---

## Formal Notation

**Probability of event A** (uniform distribution):
$$P(A) = \frac{|A|}{|\Omega|}$$
where $\Omega$ (omega) is the sample space and $|\cdot|$ denotes the size of a set.

**Complement** — the probability A does *not* occur:
$$P(A^c) = 1 - P(A)$$

**Variance** (population):
$$\sigma^2 = \frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2$$

**Standard deviation**:
$$\sigma = \sqrt{\sigma^2}$$

**Mean** (expected value for uniform discrete distribution):
$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$$

---

## Real-World Connection

- **Monitoring and alerting**: deciding whether a spike in error rate is signal or noise requires understanding variance and the Law of Large Numbers. A 10% error rate from 10 requests is very different from 10% from 10,000. High variance means you need more data before you can trust the signal.
- **Random sampling**: `random.choice()`, `np.random.randint()`, shuffling, bootstrapping — all rely on a uniform distribution over a sample space. Understanding what a sample space *is* lets you reason about what these functions are actually doing.
- **Game development**: procedural generation, loot tables, and spawn rates are all defined by probability distributions over a sample space. Standard deviation tells you how "swingy" a mechanic will feel to players.
- **Performance engineering**: benchmarking a function by running it once is meaningless. You run it many times and look at mean *and* standard deviation. A fast mean with a high std dev means unpredictable performance.

---

## Summary

- **Probability** is normalised counting: `P(A) = len(A) / len(sample_space)` for uniform distributions. It is always between 0 and 1.
- The **sample space** is the full set of possible outcomes. An **event** is any subset.
- **Law of Large Numbers**: observed frequencies converge to true probabilities as trials accumulate — but small samples are inherently noisy and should not be trusted.
- **Variance** measures average squared distance from the mean. **Standard deviation** is its square root, in the same units as the data.
- Two processes can have the same mean but completely different behaviour — variance is what distinguishes them.

**Next:** Notebook 09 — Compound Events and Independence (AND, OR, and why the gambler's fallacy is wrong).